# Company Dimension LoaderMaintains the `warehouse.dim_company` dimension table with incremental refresh capability.## PurposeTrack canonical company entities resolved from intermediate layer for consistent company references across warehouse.## Key Features* Stable surrogate keys for companies (auto-increment)* Canonical company name resolution from intermediate matching* Match confidence and method tracking* Company-to-sector mapping via dimension lookup* Alias count for fuzzy matching quality* SCD Type 1 (overwrite on change)* Idempotent: safe to re-run## Architecture**Source**: intermediate layer canonical companies and mapping (`workspace.intermediate.canonical_companies`, `workspace.intermediate.inter_company_map`)  **Target**: `workspace.warehouse.dim_company`  **Metadata**: `workspace.metadata.dim_company_refresh_log`  **Mode**: Incremental (merge new/updated companies)## Batch Processing* Tracks refresh history in metadata table* Auto-assigns surrogate keys to new companies* Updates existing companies with latest intermediate mappings* Validates data quality and sector linkage after refresh

In [0]:
dbutils.widgets.dropdown("force_full_refresh", "false", ["true", "false"], "Force Full Refresh")

FORCE_FULL_REFRESH = dbutils.widgets.get("force_full_refresh") == "true"

In [0]:
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, BooleanType, LongType, DoubleType
import json

CATALOG = "workspace"
WAREHOUSE_SCHEMA = f"{CATALOG}.warehouse"
METADATA_SCHEMA = f"{CATALOG}.metadata"
INTERMEDIATE_SCHEMA = f"{CATALOG}.intermediate"

SOURCE_MAP_TABLE = f"{INTERMEDIATE_SCHEMA}.inter_company_map"
CANONICAL_COMPANIES_TABLE = f"{INTERMEDIATE_SCHEMA}.canonical_companies"
TARGET_TABLE = f"{WAREHOUSE_SCHEMA}.dim_company"
METADATA_TABLE = f"{METADATA_SCHEMA}.dim_company_refresh_log"

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
run_timestamp = datetime.now()

print(f"Run ID: {run_id}")
print(f"Source tables: {CANONICAL_COMPANIES_TABLE}, {SOURCE_MAP_TABLE}")
print(f"Force full refresh: {FORCE_FULL_REFRESH}")

In [0]:
%sql
-- Create company dimension table if not exists
CREATE TABLE IF NOT EXISTS workspace.warehouse.dim_company (
  company_sk BIGINT NOT NULL COMMENT 'Surrogate key for company',
  company_name STRING NOT NULL COMMENT 'Display company name',
  company_name_canonical STRING NOT NULL COMMENT 'Canonical company name',
  company_match_method STRING COMMENT 'Matching method used',
  match_confidence DOUBLE COMMENT 'Match confidence score',
  alias_count INT COMMENT 'Number of company name aliases',
  sector_sk BIGINT NOT NULL COMMENT 'Foreign key to sector dimension',
  sector_name STRING COMMENT 'Sector name for reference',
  is_active BOOLEAN NOT NULL COMMENT 'Is company active',
  updated_at TIMESTAMP NOT NULL COMMENT 'Last update timestamp',
  CONSTRAINT pk_dim_company PRIMARY KEY (company_sk)
)
USING DELTA
COMMENT 'Canonical company dimension from intermediate layer';

-- Create metadata tracking table
CREATE TABLE IF NOT EXISTS workspace.metadata.dim_company_refresh_log (
  run_id STRING,
  companies_extracted INT,
  companies_inserted INT,
  companies_updated INT,
  force_full_refresh BOOLEAN,
  processed_at TIMESTAMP,
  status STRING,
  error_message STRING
)
USING DELTA
COMMENT 'Tracks company dimension refresh history';

In [0]:
print("Extracting canonical companies from intermediate layer...", end=" ")

# Load canonical companies (master list)
canonical_companies_df = spark.table(f"{INTERMEDIATE_SCHEMA}.canonical_companies")

# Load and aggregate company mapping stats
company_map_stats_df = spark.table(SOURCE_MAP_TABLE).groupBy(
    "canonical_company_id",
    "canonical_company_name"
).agg(
    F.max("normalization_confidence").alias("max_confidence"),
    F.first("normalization_method").alias("match_method"),
    F.countDistinct("enterprise_job_id").alias("alias_count")
)

# Join canonical companies with mapping stats
company_extract_df = canonical_companies_df.alias("c").join(
    company_map_stats_df.alias("m"),
    F.col("c.company_id") == F.col("m.canonical_company_id"),
    "left"
).select(
    F.col("c.canonical_company_name").alias("company_name_canonical"),
    F.col("c.canonical_company_name").alias("company_name"),
    F.coalesce(F.col("m.match_method"), F.lit("MANUAL")).alias("company_match_method"),
    F.coalesce(F.col("m.max_confidence"), F.lit(1.0)).alias("match_confidence"),
    F.coalesce(F.col("m.alias_count"), F.lit(0)).alias("alias_count"),
    F.lit(-1).cast(LongType()).alias("sector_sk"),
    F.lit(None).cast(StringType()).alias("sector_name"),
    F.lit(True).alias("is_active")
).distinct()

companies_count = company_extract_df.count()
print(f"✓ Extracted {companies_count} canonical companies")

# Get current max surrogate key
max_sk_result = spark.sql(f"SELECT COALESCE(MAX(company_sk), 0) as max_sk FROM {TARGET_TABLE}").collect()
max_sk = max_sk_result[0]['max_sk']

print(f"Current max surrogate key: {max_sk}")

In [0]:

# Define metadata schema
metadata_schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("companies_extracted", IntegerType(), True),
    StructField("companies_inserted", IntegerType(), True),
    StructField("companies_updated", IntegerType(), True),
    StructField("force_full_refresh", BooleanType(), True),
    StructField("processed_at", TimestampType(), True),
    StructField("status", StringType(), True),
    StructField("error_message", StringType(), True)
])

try:
    print(f"Processing companies into {TARGET_TABLE}...", end=" ")
    
    # Check if table schema matches expected schema FIRST
    existing_cols = [row.col_name for row in spark.sql(f"DESCRIBE {TARGET_TABLE}").collect()]
    expected_cols = ['company_sk', 'company_name', 'company_name_canonical', 'company_match_method', 'match_confidence', 'alias_count', 'sector_sk', 'sector_name', 'is_active', 'updated_at']
    schema_matches = set(existing_cols) == set(expected_cols)
    
    # Only query existing companies if schema matches
    if schema_matches:
        existing_companies = spark.sql(f"SELECT company_name_canonical, company_sk FROM {TARGET_TABLE}")
        
        # Join to assign keys (existing or new)
        from pyspark.sql.window import Window
        
        companies_with_keys = company_extract_df.alias("c").join(
            existing_companies.alias("e"),
            F.col("c.company_name_canonical") == F.col("e.company_name_canonical"),
            "left"
        )
        
        # Assign surrogate keys
        window_spec = Window.orderBy("c.company_name_canonical")
        
        companies_final = companies_with_keys.withColumn(
            "company_sk",
            F.coalesce(F.col("e.company_sk"), F.lit(max_sk) + F.row_number().over(window_spec))
        ).withColumn(
            "updated_at",
            F.lit(run_timestamp)
        ).select(
            F.col("company_sk").cast(LongType()),
            F.col("c.company_name").alias("company_name"),
            F.col("c.company_name_canonical").alias("company_name_canonical"),
            F.col("c.company_match_method").alias("company_match_method"),
            F.col("c.match_confidence").cast(DoubleType()).alias("match_confidence"),
            F.col("c.alias_count").alias("alias_count"),
            F.col("c.sector_sk").alias("sector_sk"),
            F.col("c.sector_name").alias("sector_name"),
            F.col("c.is_active").alias("is_active"),
            "updated_at"
        )
    else:
        # Schema mismatch: assign keys without joining to existing table
        from pyspark.sql.window import Window
        window_spec = Window.orderBy("company_name_canonical")
        
        companies_final = company_extract_df.withColumn(
            "company_sk",
            (F.lit(max_sk) + F.row_number().over(window_spec)).cast(LongType())
        ).withColumn(
            "updated_at",
            F.lit(run_timestamp)
        ).select(
            "company_sk",
            "company_name",
            "company_name_canonical",
            "company_match_method",
            F.col("match_confidence").cast(DoubleType()).alias("match_confidence"),
            "alias_count",
            "sector_sk",
            "sector_name",
            "is_active",
            "updated_at"
        )
    
    # Deduplicate by company_name_canonical (keep highest confidence)
    from pyspark.sql.window import Window as W
    
    dedup_window = W.partitionBy("company_name_canonical").orderBy(
        F.col("match_confidence").desc_nulls_last(),
        F.col("updated_at").desc()
    )
    
    companies_final = companies_final.withColumn(
        "row_num", F.row_number().over(dedup_window)
    ).filter(
        F.col("row_num") == 1
    ).drop("row_num")
    
    # Create temp view for merge
    companies_final.createOrReplaceTempView("companies_to_merge")
    
    if FORCE_FULL_REFRESH or not schema_matches:
        # Full refresh: drop and recreate with new schema
        if not schema_matches:
            print(f"Schema mismatch detected. Performing full refresh...")
            spark.sql(f"DROP TABLE IF EXISTS {TARGET_TABLE}")
            spark.sql(f"""
            CREATE TABLE {TARGET_TABLE} (
              company_sk BIGINT NOT NULL COMMENT 'Surrogate key for company',
              company_name STRING NOT NULL COMMENT 'Display company name',
              company_name_canonical STRING NOT NULL COMMENT 'Canonical company name',
              company_match_method STRING COMMENT 'Matching method used',
              match_confidence DOUBLE COMMENT 'Match confidence score',
              alias_count INT COMMENT 'Number of company name aliases',
              sector_sk BIGINT NOT NULL COMMENT 'Foreign key to sector dimension',
              sector_name STRING COMMENT 'Sector name for reference',
              is_active BOOLEAN NOT NULL COMMENT 'Is company active',
              updated_at TIMESTAMP NOT NULL COMMENT 'Last update timestamp',
              CONSTRAINT pk_dim_company PRIMARY KEY (company_sk)
            )
            USING DELTA
            COMMENT 'Canonical company dimension from intermediate layer'
            """)
        else:
            spark.sql(f"TRUNCATE TABLE {TARGET_TABLE}")
        
        companies_final.write.format("delta").mode("append").saveAsTable(TARGET_TABLE)
        companies_inserted = companies_final.count()
        companies_updated = 0
        print(f"✓ Full refresh: {companies_inserted} companies inserted")
    else:
        # Incremental: merge
        merge_sql = f"""
        MERGE INTO {TARGET_TABLE} target
        USING companies_to_merge source
        ON target.company_name_canonical = source.company_name_canonical
        WHEN MATCHED THEN UPDATE SET
            target.company_name = source.company_name,
            target.company_match_method = source.company_match_method,
            target.match_confidence = source.match_confidence,
            target.alias_count = source.alias_count,
            target.sector_sk = source.sector_sk,
            target.sector_name = source.sector_name,
            target.is_active = source.is_active,
            target.updated_at = source.updated_at
        WHEN NOT MATCHED THEN INSERT *
        """
        
        spark.sql(merge_sql)
        
        # Count metrics
        companies_inserted = spark.sql(f"""
            SELECT COUNT(*) as cnt FROM companies_to_merge
            WHERE company_name_canonical NOT IN (SELECT company_name_canonical FROM {TARGET_TABLE})
        """).collect()[0]['cnt']
        
        companies_updated = companies_count - companies_inserted
        
        print(f"✓ Merge complete: {companies_inserted} new, {companies_updated} updated")
    
    # Log to metadata
    metadata_data = [(
        run_id,
        companies_count,
        companies_inserted,
        companies_updated,
        FORCE_FULL_REFRESH,
        run_timestamp,
        'success',
        None
    )]
    
    metadata_record = spark.createDataFrame(metadata_data, schema=metadata_schema)
    metadata_record.write.format("delta").mode("append").saveAsTable(METADATA_TABLE)
    
    result = {
        "status": "success",
        "run_id": run_id,
        "companies_extracted": companies_count,
        "companies_inserted": companies_inserted,
        "companies_updated": companies_updated,
        "target_table": TARGET_TABLE,
        "metadata_table": METADATA_TABLE
    }
    
    print(json.dumps(result, indent=2))
    
except Exception as e:
    error_msg = str(e)
    print(f"✗ Error: {error_msg}")
    
    # Log failure to metadata
    metadata_data = [(
        run_id,
        companies_count if 'companies_count' in locals() else 0,
        0,
        0,
        FORCE_FULL_REFRESH,
        run_timestamp,
        'failed',
        error_msg
    )]
    
    metadata_record = spark.createDataFrame(metadata_data, schema=metadata_schema)
    metadata_record.write.format("delta").mode("append").saveAsTable(METADATA_TABLE)
    
    raise

In [0]:
%sql
-- Validate company dimension
SELECT 
  COUNT(*) as total_companies,
  COUNT(DISTINCT sector_sk) as sectors_represented,
  COUNT(DISTINCT company_match_method) as match_methods,
  AVG(match_confidence) as avg_match_confidence,
  AVG(alias_count) as avg_alias_count,
  SUM(CASE WHEN is_active THEN 1 ELSE 0 END) as active_companies
FROM workspace.warehouse.dim_company;


In [0]:
%sql
-- Sample companies with sector and match info
SELECT 
  company_sk,
  company_name,
  company_name_canonical,
  sector_name,
  company_match_method,
  match_confidence,
  alias_count,
  is_active,
  updated_at
FROM workspace.warehouse.dim_company
ORDER BY match_confidence DESC, company_name
LIMIT 25;

In [0]:
%sql
-- Show refresh history
SELECT 
  run_id,
  companies_extracted,
  companies_inserted,
  companies_updated,
  force_full_refresh,
  processed_at,
  status
FROM workspace.metadata.dim_company_refresh_log
ORDER BY processed_at DESC
LIMIT 10;